# Regressió ML amb Sklearn

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Càrrega de dades

Aquestes dades han d'estar netes i preparades per aplicar ML.

En aquest exemple de `tips`, volem predir la propina (`tip`) que pagarà un client.

In [2]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## Pipeline de Feature Engineering

Defineix els passos de preprocessament per als models que ho necessiten.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [4]:
onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

In [5]:
onehot_features = ['sex', 'smoker', 'day', 'time']

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_pipeline, onehot_features),
    ],
    remainder='passthrough',
)

## Train / Test

- Podem emular dades no vistes dividint les nostres dades en conjunts de **train** i de **test**.
    - Usem el conjunt d'entrenament per ajustar (entrenar) el nostre model.
    - Usem el conjunt de test per avaluar el rendiment del model amb dades no vistes.
- Les mides de test habituals són del 10-30% de les dades inicials.
    - Els conjunts de dades grans poden requerir un percentatge menor (p. ex., 10-20%).
    - Els conjunts de dades petits poden necessitar un conjunt de test més gran per garantir que sigui representatiu (p. ex., 30-40%).

In [6]:
from sklearn.model_selection import train_test_split, cross_validate, KFold

In [7]:
# Crear la variable X amb les característiques predictores (sense l'objectiu!)
X = df.drop('tip', axis=1)

# Separar l'objectiu en la variable `y`
y = df['tip']

# Divisió Entrenament / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% de les dades en el test
    shuffle=True,
    random_state=42
)

### Cross-validation

Definim l'objecte per aplicar cross-validation. Un cop definit, tots els models l'utilitzaran i, per tant, tots aplicaran el mateix tipus de CV. Això és útil perquè si apliquem un CV diferent a tots els models, no podríem comparar els resultats entre ells.

In [8]:
# Definim CV
kfold = KFold(10, shuffle=True, random_state=42)

## Entrenament del model

In [9]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

### Baseline

- Entrena i avalua el rendiment d'un model de referència (baseline).
- Proporciona una mesura inicial del rendiment.
- Qualsevol model que desenvolupis hauria de superar aquesta línia base simple.

In [10]:
from sklearn.dummy import DummyRegressor

In [11]:
# El baseline predirà sempre el valor mitjà de la columna target
bl = DummyRegressor(strategy='mean')

# Entrenar
bl_cv = cross_validate(
    bl,
    X_train,
    y_train,
    cv=kfold,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)

In [12]:
bl_cv

{'fit_time': array([0.00407791, 0.00658536, 0.00698805, 0.00515842, 0.00738597,
        0.00296187, 0.00546932, 0.007936  , 0.00308633, 0.00394678]),
 'score_time': array([0.00117493, 0.00250602, 0.00228643, 0.00310946, 0.0015173 ,
        0.00084114, 0.00079536, 0.00206947, 0.00104904, 0.00167871]),
 'test_score': array([-1.34441429, -0.75078286, -0.90314286, -0.99909143, -0.6495    ,
        -1.00163278, -1.07003589, -1.15882177, -1.24207237, -1.4874372 ]),
 'train_score': array([-1.02443102, -1.09517714, -1.07723102, -1.05826351, -1.09793894,
        -1.05050232, -1.05916581, -1.03905992, -1.02695313, -0.99436467])}

In [13]:
avg_train_mae = -bl_cv['train_score'].mean()
avg_val_mae = -bl_cv['test_score'].mean()  # Validation score

print(f'Avg. train MAE: {avg_train_mae:.1f}')
print(f'Avg. val. MAE: {avg_val_mae:.1f}')

Avg. train MAE: 1.1
Avg. val. MAE: 1.1


### Regressió lineal

- Cada vegada que entrenem la regressió lineal,
- el model aprèn els coeficients $w_i$ (en funció de les característiques $x_i$):

    $\hat{y} = w_0 + w_1 \cdot x_1 + w_2 \cdot x_2 + \text{...} + w_n \cdot x_n$


- de manera que les prediccions $\hat{y}$ minimitzin $MSE(y, \hat{y})$.

In [14]:
from sklearn.linear_model import LinearRegression

In [15]:
# Definim una Pipeline que primer processarà les dades i llavors aplicarà LR
lr = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LinearRegression(n_jobs=-1))
])

# Entrenar
lr_cv = cross_validate(
    lr,
    X_train,
    y_train,
    cv=kfold,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    n_jobs=-1
)

In [16]:
avg_train_mae = -lr_cv['train_score'].mean()
avg_val_mae = -lr_cv['test_score'].mean()

print(f'Avg. train MAE: {avg_train_mae:.1f}')
print(f'Avg. val. MAE: {avg_val_mae:.1f}')

Avg. train MAE: 0.8
Avg. val. MAE: 0.8
